In [0]:
/**
Version 1.1.0
Desc: captures rejected annotation rows (missing, invalid, or duplicate) for transparency
created by CPJ on 9/16/2026
**/

create or refresh materialized view vehicle_project.silver.clean_vehicle_annotations_rejected
as
select
    *,
    -- Missing records
    case
        when filename is null then 'missing_filename'
        when class_id is null then 'missing_class_id'
        when x_center is null then 'missing_x_center'
        when y_center is null then 'missing_y_center'
        when width is null then 'missing_width'
        when height is null then 'missing_height'
        else null
    end as missing,

    -- Invalid records
    case
        when class_id not between 0 and 13 then 'invalid_class_id'
        when split not in ('train', 'valid', 'test') then 'invalid_split'
        when x_center not between 0 and 1 then 'invalid_x_center'
        when y_center not between 0 and 1 then 'invalid_y_center'
        when width not between 0 and 1 then 'invalid_width'
        when height not between 0 and 1 then 'invalid_height'
        when width <= 0 or height <= 0 then 'invalid_bbox_area'
        when x_center - width / 2 < 0 or x_center + width / 2 > 1 then 'invalid_bbox_x'
        when y_center - height / 2 < 0 or y_center + height / 2 > 1 then 'invalid_bbox_y'
        else null
    end as invalid_bbox,

    -- Duplicate records
    case
        when row_number() over (
            partition by filename, class_id, x_center, y_center, width, height
            order by _ingested_at
        ) > 1 then 'duplicate_detection'
        else null
    end as duplicate_reason

from vehicle_project.bronze.raw_vehicle_annotations
;